# Phase 4 - Step 1: Court Judgment Prediction (CJPE) data exploration

Goal: understand the CJPE data before designing the models. No GPU needed.

1. download `Phase_4_CJPE/` from Hugging Face (skipped if already on disk)
2. file format and data checks (duplicates, empty texts, leaks between splits)
3. label balance (Macro-F1 -> both classes count equally)
4. document lengths in words and in **bpe32k tokens** -> how much of a case fits in 1536 tokens, and how long one epoch would take
5. does length or case type predict the outcome?
6. what the documents look like (start, end, headings, outcome words)
7. **which part of the document carries the signal?** a tiny word-based classifier (Naive Bayes) trained on the first 512 words, the last 512 words, both, or the whole case

Results are cached in `experiments/phase4/p4_01_data/` (re-running loads them).


In [1]:
import json, os, re, sys, hashlib, collections, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# project root = first folder upwards that contains .gitignore (same as all earlier notebooks)
ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from ner_utils import CHAR_MAP          # the Phase-1/2 character cleaning table

RAW_DIR = ROOT / "data" / "raw"
P4 = RAW_DIR / "Phase_4_CJPE"
OUT = ROOT / "experiments" / "phase4" / "p4_01_data"
OUT.mkdir(parents=True, exist_ok=True)
TOK_PATH = ROOT / "data" / "tokenizer" / "bpe32k.json"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("project root:", ROOT)

project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project


c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download the Phase 4 files (skipped if already on disk)


In [2]:
FILES = {"train": P4 / "Train" / "train.json", "dev": P4 / "Train" / "dev.json", "test": P4 / "Test" / "test.json"}
if all(p.exists() for p in FILES.values()):
    print("Phase 4 data already downloaded -> skipping download")
else:
    from huggingface_hub import snapshot_download
    snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                      allow_patterns="Phase_4_CJPE/*", local_dir=str(RAW_DIR))
for p in sorted(P4.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(P4)):25s} {p.stat().st_size / 1e6:9.2f} MB")
extra = [p for p in P4.rglob("*") if p.is_file() and p not in FILES.values()]
if extra:
    print("NOTE: files not used by this notebook:", [str(p.relative_to(P4)) for p in extra])

Phase 4 data already downloaded -> skipping download
Test\test.json                35.27 MB
Train\dev.json                44.96 MB
Train\train.json             736.65 MB


## 2. Load the files and look at the format

The doc says each example has `id`, `text` and (train/dev) a binary label, but not the exact key names
or whether the file is a list or a dict. The loader below finds them automatically and prints what it found,
so we see the real format before trusting it:

- the **label key** = the first of `label, labels, y, outcome, decision, judgment, pred` that is present
- if `text` is a list of sentences/paragraphs, it is joined with newlines


In [3]:
LABEL_KEYS = ["label", "labels", "y", "outcome", "decision", "judgment", "judgement", "pred"]
TEXT_KEYS = ["text", "document", "case_text", "judgment_text", "content"]

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def to_doc_list(data):
    # a list of records, or a dict {id: record}
    if isinstance(data, list):
        return [dict(d) for d in data]
    if isinstance(data, dict):
        out = []
        for k, v in data.items():
            d = dict(v) if isinstance(v, dict) else {"text": v}
            d.setdefault("id", k)
            out.append(d)
        return out
    raise TypeError(f"unexpected JSON type {type(data)}")

def find_key(record, candidates):
    return next((k for k in candidates if k in record), None)

def normalise_record(d, text_key, label_key):
    text = d[text_key]
    if isinstance(text, list):                 # list of sentences/paragraphs -> one string
        text = "\n".join(map(str, text))
    lab = d.get(label_key) if label_key else None
    if isinstance(lab, list) and len(lab) == 1:
        lab = lab[0]
    return {"id": str(d["id"]), "text": str(text), "label": None if lab is None else int(lab)}

SPLITS, FORMAT = {}, {}
for name, path in FILES.items():
    raw = read_json(path)
    recs = to_doc_list(raw)
    first = recs[0]
    tkey, lkey = find_key(first, TEXT_KEYS), find_key(first, LABEL_KEYS)
    assert tkey, f"{name}: no text field among {list(first.keys())}"
    FORMAT[name] = {"file_type": type(raw).__name__, "keys": list(first.keys()), "text_key": tkey,
                    "text_type": type(first[tkey]).__name__, "label_key": lkey}
    SPLITS[name] = [normalise_record(d, tkey, lkey) for d in recs]
    print(f"===== {name}: {len(recs):,} records | {FORMAT[name]}")
    d0 = SPLITS[name][0]
    print(f"  first id={d0['id']!r} label={d0['label']} | text starts: {d0['text'][:200]!r}\n")

===== train: 37,387 records | {'file_type': 'list', 'keys': ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'], 'text_key': 'text', 'text_type': 'str', 'label_key': 'label'}
  first id='2020_1' label=0 | text starts: ' Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of the High Court 1 dismissing S.T. No.192 of 2014 and other companynected matters'

===== dev: 1,988 records | {'file_type': 'list', 'keys': ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'], 'text_key': 'text', 'text_type': 'str', 'label_key': 'label'}
  first id='1989_75' label=0 | text starts: 'civil appellate jurisdiction civil appeal number 1709\nof\n1988.\nfrom the judgment and order dated 9.10.1987 of t\nhe\ncentral administrative tribunal chandigarh in appln. t.\n--\n1055 of 1986.\nwith\n slp  c'

===== test: 1,517 records | {'file_type': 'list', 'keys': ['id', 'text', 'label', 'ex

## 3. Data checks

- duplicate ids, empty or very short texts
- labels must be 0/1 in train and dev; test must have no labels
- line breaks (paragraph structure) and non-ASCII characters (Phase 1 trained on cleaned ASCII text)
- **duplicate texts inside and across splits** (a train case that reappears in dev/test would inflate scores;
  the same text with different labels = label noise)


In [4]:
def text_key(t):
    # whitespace-insensitive, case-insensitive fingerprint of a document
    return hashlib.md5(re.sub(r"\s+", " ", t).strip().lower().encode()).hexdigest()

rows, by_hash = [], collections.defaultdict(list)
for name, docs in SPLITS.items():
    ids = [d["id"] for d in docs]
    labs = [d["label"] for d in docs]
    n_chars = sum(len(d["text"]) for d in docs)
    non_ascii = sum(sum(ord(c) > 127 for c in d["text"]) for d in tqdm(docs, desc=f"checking {name}", leave=False))
    for d in docs:
        by_hash[text_key(d["text"])].append((name, d["id"], d["label"]))
    rows.append({
        "split": name, "docs": len(docs), "dup_ids": len(ids) - len(set(ids)),
        "labelled": sum(l is not None for l in labs),
        "label_values": sorted(set(l for l in labs if l is not None)),
        "empty": sum(not d["text"].strip() for d in docs),
        "<100 words": sum(len(d["text"].split()) < 100 for d in docs),
        "docs_with_newlines_%": round(100 * np.mean(["\n" in d["text"] for d in docs]), 1),
        "lines/doc (mean)": round(float(np.mean([d["text"].count("\n") + 1 for d in docs])), 1),
        "non_ascii_%": round(100 * non_ascii / max(n_chars, 1), 3),
        "signature_stamps": sum("Signature Not Verified" in d["text"] for d in docs),
    })
checks = pd.DataFrame(rows).set_index("split")
display(checks)

dup_groups = [v for v in by_hash.values() if len(v) > 1]
pair_counts = collections.Counter(tuple(sorted({s for s, _, _ in v})) for v in dup_groups)
conflicting = [v for v in dup_groups if len({l for _, _, l in v if l is not None}) > 1]
print(f"duplicate texts: {len(dup_groups)} groups; which splits they span: {dict(pair_counts)}")
print(f"  duplicates with CONFLICTING labels: {len(conflicting)}")
for v in dup_groups[:5]:
    print("  example group:", v[:4])
ids_shared = {a + "&" + b: len({d["id"] for d in SPLITS[a]} & {d["id"] for d in SPLITS[b]})
              for a, b in [("train", "dev"), ("train", "test"), ("dev", "test")]}
print("ids shared between splits:", ids_shared)

,docs,dup_ids,labelled,label_values,empty,<100 words,docs_with_newlines_%,lines/doc (mean),non_ascii_%,signature_stamps
split,,,,,,,,,,
train,37387,5082,37387,"[0, 1]",0,572,0.0,1.0,0.0,1583
dev,1988,994,1988,"[0, 1]",0,0,100.0,374.6,0.0,0
test,1517,0,1517,"[0, 1]",0,1,98.9,379.7,0.0,0


duplicate texts: 6708 groups; which splits they span: {('train',): 5714, ('dev',): 994}
  duplicates with CONFLICTING labels: 0
  example group: [('train', '2019_890', 1), ('train', '2019_890', 1)]
  example group: [('train', '2019_155', 0), ('train', '2019_172', 0)]
  example group: [('train', '2017_43', 0), ('train', '2017_554', 0)]
  example group: [('train', '2016_232', 0), ('train', '2016_681', 0)]
  example group: [('train', '2016_293', 0), ('train', '2016_299', 0)]
ids shared between splits: {'train&dev': 0, 'train&test': 0, 'dev&test': 0}


## 4. Label balance

Macro-F1 = average of the F1 of class 0 and the F1 of class 1, so the minority class counts as much as the
majority class. The baselines below show what a model gets **without reading anything**.


In [5]:
def f1_scores(gold, pred):
    # macro-F1, macro precision and macro recall over the two classes (same as sklearn average="macro")
    gold, pred = np.asarray(gold), np.asarray(pred)
    out = {}
    for c in (0, 1):
        tp = int(((pred == c) & (gold == c)).sum()); fp = int(((pred == c) & (gold != c)).sum())
        fn = int(((pred != c) & (gold == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        out[c] = (p, r, 2 * p * r / (p + r) if p + r else 0.0)
    return {"macro_f1": round((out[0][2] + out[1][2]) / 2, 4), "macro_p": round((out[0][0] + out[1][0]) / 2, 4),
            "macro_r": round((out[0][1] + out[1][1]) / 2, 4), "f1_0": round(out[0][2], 4), "f1_1": round(out[1][2], 4),
            "acc": round(float((gold == pred).mean()), 4)}

LAB = [n for n in SPLITS if all(d["label"] is not None for d in SPLITS[n])]
bal = pd.DataFrame({n: collections.Counter(d["label"] for d in SPLITS[n]) for n in LAB}).fillna(0).astype(int)
bal.loc["% label 1"] = (100 * bal.loc[1] / bal.sum()).round(1)
display(bal)

if "dev" in LAB:
    gold = [d["label"] for d in SPLITS["dev"]]
    rng = np.random.default_rng(0)
    p1 = np.mean([d["label"] for d in SPLITS["train"]])
    base = {"always 0": f1_scores(gold, [0] * len(gold)), "always 1": f1_scores(gold, [1] * len(gold)),
            "random (train rate)": f1_scores(gold, (rng.random(len(gold)) < p1).astype(int))}
    display(pd.DataFrame(base).T.rename_axis("dev baseline"))

,train,dev,test
0,22067.0,994.0,755.0
1,15320.0,994.0,762.0
% label 1,41.0,50.0,50.2


,macro_f1,macro_p,macro_r,f1_0,f1_1,acc
dev baseline,,,,,,
always 0,0.3333,0.250,0.500,0.6667,0.0000,0.500
always 1,0.3333,0.250,0.500,0.0000,0.6667,0.500
random (train rate),0.4974,0.501,0.501,0.5399,0.4549,0.501


## 5. Lengths in words and characters


In [6]:
def pct_row(x):
    x = np.asarray(x)
    return {"mean": round(float(x.mean()), 1), "p10": int(np.percentile(x, 10)), "p50": int(np.percentile(x, 50)),
            "p90": int(np.percentile(x, 90)), "p99": int(np.percentile(x, 99)), "max": int(x.max())}

WORDS = {n: [len(d["text"].split()) for d in docs] for n, docs in SPLITS.items()}
display(pd.DataFrame({n: pct_row(w) for n, w in WORDS.items()}).T.rename_axis("words per document"))
display(pd.DataFrame({n: pct_row([len(d["text"]) for d in docs]) for n, docs in SPLITS.items()}).T
        .rename_axis("characters per document"))

,mean,p10,p50,p90,p99,max
words per document,,,,,,
train,3276.7,408.0,2192.0,6575.0,19403.0,412606.0
dev,3752.5,1200.0,2821.0,6933.0,19912.0,42198.0
test,3850.2,1212.0,2940.0,7302.0,18235.0,40487.0


,mean,p10,p50,p90,p99,max
characters per document,,,,,,
train,19531.3,2453.0,13023.0,38996.0,115746.0,2492701.0
dev,22110.3,7094.0,16700.0,40495.0,117842.0,260198.0
test,22697.9,7157.0,17320.0,42859.0,108673.0,233000.0


## 6. Lengths in bpe32k tokens (cached)

Each case is cleaned exactly like the Phase-1 training text (signature stamps removed, special characters
mapped to ASCII, non-ASCII dropped, all whitespace -> single spaces), then tokenised with my bpe32k tokenizer.

From the counts we get:

- how many cases fit in one 1536-token window (the Phase-1 context)
- how many windows (1536 tokens, stride 1024, as in Phases 2-3) a whole case needs
- **a rough time per epoch** for each way of reading a case, using speeds measured in Phase 3
  (A2 full fine-tuning: ~675k tokens in 46 s ≈ 14.7k tokens/s training; ~98.6k tokens/s inference).


In [7]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(str(TOK_PATH))
print("tokenizer:", TOK_PATH.relative_to(ROOT), "| vocab", tok.get_vocab_size())

SIG_RE = re.compile(r"Signature Not Verified.{0,300}?Reason:", re.S)

def clean_text(t):
    # Phase-1 cleaning: stamps out, special chars -> ASCII, drop other non-ASCII, single spaces
    t = SIG_RE.sub(" ", t)
    t = "".join(CHAR_MAP.get(c, c) for c in t)
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", t).strip()

cache = OUT / "token_counts.json"
if cache.exists():
    TOKS = read_json(cache)
    print("loaded cached token counts")
else:
    TOKS = {}
    for name, docs in SPLITS.items():
        counts = []
        for i in tqdm(range(0, len(docs), 256), desc=f"tokenising {name}"):
            batch = [" " + clean_text(d["text"]) for d in docs[i:i + 256]]
            counts += [len(e.ids) for e in tok.encode_batch(batch, add_special_tokens=False)]
        TOKS[name] = counts
    cache.write_text(json.dumps(TOKS))

def n_windows(n, L=1536, stride=1024):
    return 1 if n <= L else 1 + int(np.ceil((n - L) / stride))

rows = {}
for name, t in TOKS.items():
    t = np.asarray(t)
    rows[name] = {**pct_row(t), "tok/word": round(t.sum() / max(sum(WORDS[name]), 1), 3),
                  "<=512 %": round(100 * (t <= 512).mean(), 1), "<=1536 %": round(100 * (t <= 1536).mean(), 1),
                  "<=3072 %": round(100 * (t <= 3072).mean(), 1), "<=6144 %": round(100 * (t <= 6144).mean(), 1),
                  "windows/doc": round(float(np.mean([n_windows(x) for x in t])), 2),
                  "total M tokens": round(t.sum() / 1e6, 1)}
display(pd.DataFrame(rows).T.rename_axis("bpe32k tokens per document"))

TRAIN_TPS, INFER_TPS = 14_700, 98_600   # tokens/s, measured in Phase 3 (A2 full FT)
def tokens_read(n, strategy):
    # how many tokens the model reads for one case of n tokens
    if strategy == "whole case (all windows)":       # overlapping windows -> reads more than n
        return n if n <= 1536 else 1536 * n_windows(n)
    if strategy == "last 512 tokens":
        return min(n, 512)
    return min(n, 1536)                               # last 1536, or first 768 + last 768

STRATEGIES = ["whole case (all windows)", "first 768 + last 768", "last 1536 tokens", "last 512 tokens"]
est = {}
for s in STRATEGIES:
    tr_tok = sum(tokens_read(n, s) for n in TOKS["train"])
    ev_tok = sum(tokens_read(n, s) for sp in ("dev", "test") for n in TOKS[sp])
    est[s] = {"M tokens read / epoch": round(tr_tok / 1e6, 1),
              "train min / epoch (full FT)": round(tr_tok / TRAIN_TPS / 60, 1),
              "dev+test inference min": round(ev_tok / INFER_TPS / 60, 1)}
est = pd.DataFrame(est).T
display(est.rename_axis("reading strategy (train split)"))

tokenizer: data\tokenizer\bpe32k.json | vocab 32000
loaded cached token counts


,mean,p10,p50,p90,p99,max,tok/word,<=512 %,<=1536 %,<=3072 %,<=6144 %,windows/doc,total M tokens
bpe32k tokens per document,,,,,,,,,,,,,
train,3942.2,495.0,2644.0,7887.0,23212.0,488398.0,1.203,10.4,30.1,56.8,84.2,3.94,147.4
dev,4490.4,1471.0,3406.0,8259.0,23648.0,52674.0,1.197,0.9,11.0,43.2,80.1,4.40,8.9
test,4610.1,1473.0,3523.0,8576.0,21424.0,51647.0,1.197,1.3,10.9,42.0,78.9,4.52,7.0


,M tokens read / epoch,train min / epoch (full FT),dev+test inference min
reading strategy (train split),,,
whole case (all windows),217.7,246.9,4.0
first 768 + last 768,48.7,55.2,0.9
last 1536 tokens,48.7,55.2,0.9
last 512 tokens,18.2,20.7,0.3


## 7. Does length or case type predict the outcome?

If longer cases were mostly "1", a model could cheat by counting tokens. A **length-only baseline** (one threshold on
the number of tokens, chosen on train) tells us how much signal is in the length alone.
Case type (civil vs criminal) is read from the first 1,000 characters.


In [8]:
tr = pd.DataFrame({"tokens": TOKS["train"], "label": [d["label"] for d in SPLITS["train"]]})
display(tr.groupby("label")["tokens"].describe().round(0))
tr["length quartile"] = pd.qcut(tr["tokens"], 4, labels=["Q1 short", "Q2", "Q3", "Q4 long"])
display(tr.groupby("length quartile", observed=True)["label"].agg(["count", "mean"]).rename(columns={"mean": "rate of 1"}).round(3))

# length-only baseline: predict 1 if tokens > threshold (or < threshold), best macro-F1 on TRAIN, then score on DEV
best = (0, None, None)
for q in np.percentile(tr["tokens"], np.arange(5, 96, 5)):
    for sign in (1, -1):
        pred = (sign * tr["tokens"] > sign * q).astype(int)
        s = f1_scores(tr["label"], pred)["macro_f1"]
        best = max(best, (s, q, sign), key=lambda x: x[0])
_, thr, sign = best
dev_pred = (sign * np.asarray(TOKS["dev"]) > sign * thr).astype(int)
LENGTH_BASELINE = {"threshold": float(thr), "direction": ">" if sign == 1 else "<",
                   **f1_scores([d["label"] for d in SPLITS["dev"]], dev_pred)}
print("length-only baseline (dev):", LENGTH_BASELINE)

def case_type(t):
    h = t[:1000].lower()
    if "criminal" in h: return "criminal"
    if "civil" in h: return "civil"
    if "writ" in h: return "writ"
    if "tax" in h or "income" in h: return "tax"
    return "other/unknown"

ct = pd.DataFrame({"type": [case_type(d["text"]) for d in SPLITS["train"]], "label": tr["label"]})
display(ct.groupby("type")["label"].agg(["count", "mean"]).rename(columns={"mean": "rate of 1"}).round(3))

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,22067.0,4042.0,8065.0,52.0,1210.0,2630.0,4701.0,462318.0
1,15320.0,3799.0,6379.0,51.0,1342.0,2658.0,4618.0,488398.0


,count,rate of 1
length quartile,,
Q1 short,9356,0.386
Q2,9339,0.429
Q3,9347,0.421
Q4 long,9345,0.403


length-only baseline (dev): {'threshold': 2950.0, 'direction': '>', 'macro_f1': 0.4587, 'macro_p': 0.4616, 'macro_r': 0.4628, 'f1_0': 0.4119, 'f1_1': 0.5056, 'acc': 0.4628}


,count,rate of 1
type,,
civil,13823,0.403
criminal,5602,0.438
other/unknown,12043,0.422
tax,1369,0.394
writ,4550,0.371


## 8. What the documents look like

Start and end of two training cases per class (line breaks shown as ⏎), the most common short lines
(probably headings such as `JUDGMENT`), and how often typical section markers appear in each class.


In [9]:
def show(t, n=450):
    return t.replace("\n", " ⏎ ")[:n]

for lab in (0, 1):
    ex = [d for d in SPLITS["train"] if d["label"] == lab][:2]
    for d in ex:
        print(f"===== label {lab} | id {d['id']} | {len(d['text'].split()):,} words")
        print("START:", show(d["text"]))
        print("END:  ", show(d["text"][-450:]), "\n")

# short lines that repeat across many documents = headings / boilerplate
line_df = collections.Counter()
for d in SPLITS["train"]:
    lines = {re.sub(r"\s+", " ", l).strip() for l in d["text"].split("\n")}
    line_df.update(l for l in lines if 0 < len(l.split()) <= 6)
print("most common short lines (document frequency, train):")
display(pd.DataFrame(line_df.most_common(30), columns=["line", "docs"]))

MARKERS = ["JUDGMENT", "ORDER", "FACTS", "Leave granted", "learned counsel", "High Court", "Trial Court",
           "appellant", "respondent", "Section", "Article", "we are of the opinion", "In the result"]
tbl = {}
for lab in (0, 1):
    docs = [d["text"] for d in SPLITS["train"] if d["label"] == lab]
    tbl[f"label {lab} (% docs)"] = {m: round(100 * np.mean([m.lower() in t.lower() for t in docs]), 1) for m in MARKERS}
display(pd.DataFrame(tbl))

===== label 0 | id 2020_1 | 43,323 words
START:  Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of the High Court 1 dismissing S.T. No.192 of 2014 and other companynected matters and thereby affirming the decision of the Single Judge of the High Court passed on 12.03.2014 in Writ Petition No.20650  W  of 2013 which in turn had found Sections 8, 10, 11 and 12 of the West Bengal Madrasah Service Commission Act, 2008  the Commi
END:   the High Court and dismiss Writ Petition No.20650 W  of 2013 and other companynected matters. We also hold Sections 8, 10, 11 and 12 of the Commission Act to be valid and companystitutional. In the end, we declare all numberinations made by the Commission in pursuance of the provisions of the Commission Act to be valid and Civil Appeal No.5808 of 2017 SK. MD. Rafique vs. Managing Committee, companytai Rahamania High Madrasah and Others operative. 

===== label 0 | id 2020_2 | 21,035 words


,line,docs


,label 0 (% docs),label 1 (% docs)
JUDGMENT,82.2,82.8
ORDER,89.0,89.9
FACTS,61.8,64.5
Leave granted,23.2,33.3
learned counsel,10.7,9.2
High Court,87.2,88.2
Trial Court,14.2,15.4
appellant,82.1,87.9
respondent,77.4,83.4
Section,76.4,76.8


## 9. Outcome words near the end (leak check)

The original dataset removed the final decision from each judgment. If words like _allowed_ / _dismissed_
still appear in the **last 200 words**, the end of the document is a strong (maybe too easy) signal.


In [10]:
OUTCOME = ["allowed", "dismissed", "set aside", "quashed", "upheld", "restored", "remanded", "affirmed",
           "no merit", "without merit", "accordingly", "costs"]
tail = {}
for lab in (0, 1):
    docs = [" ".join(d["text"].split()[-200:]).lower() for d in SPLITS["train"] if d["label"] == lab]
    tail[f"label {lab}: % docs"] = {w: round(100 * np.mean([w in t for t in docs]), 1) for w in OUTCOME}
tail = pd.DataFrame(tail)
tail["ratio 1/0"] = (tail.iloc[:, 1] / tail.iloc[:, 0].replace(0, np.nan)).round(2)
display(tail.rename_axis("word in last 200 words"))

,label 0: % docs,label 1: % docs,ratio 1/0
word in last 200 words,,,
allowed,7.2,19.9,2.76
dismissed,18.2,7.7,0.42
set aside,7.2,25.9,3.60
quashed,1.3,3.1,2.38
upheld,2.2,1.9,0.86
restored,0.7,3.3,4.71
remanded,0.6,1.3,2.17
affirmed,1.5,1.6,1.07
no merit,0.0,0.0,NaN


## 10. Which part of the case carries the signal? (word-based Naive Bayes, cached)

Before spending GPU hours, a cheap model tells us which part of a judgment predicts the outcome.

**Binarised Naive Bayes** (Wang & Manning 2012), explained simply:

- each document = the **set** of words in it (lower-case, letters only)
- for every word w: how often it appears in class-1 documents vs class-0 documents ->
  `r[w] = log( P(w | class 1) / P(w | class 0) )` (positive = typical for 1)
- score of a document = sum of `r[w]` over its words + log(prior ratio); predict 1 if score > 0

Same classifier, four inputs: **first 512 words**, **last 512 words**, **first 256 + last 256**, **whole case**.
Trained on train, scored on dev. This is an analysis tool, not a submission (Phase 4 must use my Phase-1 models).


In [11]:
WORD_RE = re.compile(r"[a-z]+")

def section(t, part):
    w = t.split()
    if part == "first 512": w = w[:512]
    elif part == "last 512": w = w[-512:]
    elif part == "first 256 + last 256": w = w[:256] + w[-256:] if len(w) > 512 else w
    return set(WORD_RE.findall(" ".join(w).lower()))

def train_nb(sets, labels, min_df=5, alpha=1.0):
    df = {0: collections.Counter(), 1: collections.Counter()}
    for s, y in zip(sets, labels):
        df[y].update(s)
    vocab = [w for w in set(df[0]) | set(df[1]) if df[0][w] + df[1][w] >= min_df]
    p0 = np.array([df[0][w] for w in vocab], float) + alpha
    p1 = np.array([df[1][w] for w in vocab], float) + alpha
    r = np.log(p1 / p1.sum()) - np.log(p0 / p0.sum())
    n1 = sum(labels); prior = np.log(n1 / (len(labels) - n1))
    return {w: v for w, v in zip(vocab, r)}, prior

def predict_nb(model, sets):
    r, prior = model
    return np.array([int(sum(r.get(w, 0.0) for w in s) + prior > 0) for s in sets])

nb_cache = OUT / "nb_sections.json"
if nb_cache.exists():
    NB = read_json(nb_cache)
    print("loaded cached Naive Bayes results")
else:
    NB = {"results": {}, "top_words": {}}
    ytr = [d["label"] for d in SPLITS["train"]]; ydev = [d["label"] for d in SPLITS["dev"]]
    for part in tqdm(["first 512", "last 512", "first 256 + last 256", "whole case"], desc="sections"):
        # generator: one training case at a time (train.json is ~740 MB, a list of all word sets would need GBs of RAM)
        Str = (section(d["text"], part) for d in tqdm(SPLITS["train"], desc=f"  {part} train", leave=False))
        Sdv = [section(d["text"], part) for d in SPLITS["dev"]]
        model = train_nb(Str, ytr)
        NB["results"][part] = f1_scores(ydev, predict_nb(model, Sdv))
        top = sorted(model[0].items(), key=lambda kv: kv[1])
        NB["top_words"][part] = {"class 0": [w for w, _ in top[:25]], "class 1": [w for w, _ in top[-25:][::-1]]}
    nb_cache.write_text(json.dumps(NB, indent=1))

display(pd.DataFrame(NB["results"]).T.rename_axis("Naive Bayes on ... (dev)"))
for part in ("last 512", "whole case"):
    print(f"\nmost class-specific words ({part}):")
    for c, ws in NB["top_words"][part].items():
        print(f"  {c}: {', '.join(ws)}")

loaded cached Naive Bayes results


,macro_f1,macro_p,macro_r,f1_0,f1_1,acc
Naive Bayes on ... (dev),,,,,,
first 512,0.5240,0.5729,0.5543,0.6442,0.4038,0.5543
last 512,0.5862,0.6246,0.6036,0.6711,0.5013,0.6036
first 256 + last 256,0.5509,0.6236,0.5855,0.6756,0.4262,0.5855
whole case,0.6030,0.6063,0.6046,0.6282,0.5779,0.6046



most class-specific words (last 512):
  class 0: impartible, malign, twentieth, quintal, pleases, benjamin, numberparticulars, vastly, develops, iraq, thakore, limaye, ariff, numberanalogy, sitabai, garbage, dialogue, imple, encounters, speedier, friction, ghatwali, beneficient, vie, maladministration
  class 1: indictment, mation, umarani, fema, prevaricating, infor, soaps, jasmer, bhada, companytumacy, silentio, emit, overdue, npa, fulcrum, numberteacher, disadvantaged, codes, leap, wheeling, stacking, penalises, statewise, satis, tacitly

most class-specific words (whole case):
  class 0: bryden, bogie, adop, jeewanlal, abolitionists, padmavati, ambalam, lavu, clogging, numberjusticiable, nambiars, rudeness, misser, brevitys, numberadic, maniar, cigar, numberallocation, aeronautical, cocoanut, karsanbhai, numbernegative, sultation, roig, symbolises
  class 1: polailal, fijji, logendranath, maurusi, mvc, milkiyat, determinatively, slovenly, pukhta, tameside, bojjappa, parse, abdus, 

## 11. Train vs dev/test style, and duplicates (added after the first run)

The first run showed that dev/test are **all lower-case, hard-wrapped old cases** (1950s-1990s), and that every dev case
appears **twice**. This cell measures the same things for train:

- writing style: share of cases with < 1% upper-case letters, share with line breaks, `-\n` word wraps
- decade of each case (from the id, e.g. `1986_397` -> 1980s) and the rate of label 1 per decade and per style
- how many **unique** cases remain after removing exact duplicate texts (and their label balance)


In [12]:
def style(t):
    letters = sum(c.isalpha() for c in t)
    upper = sum(c.isupper() for c in t)
    return "lower" if upper < 0.01 * max(letters, 1) else "cased"

def decade(i):
    y = str(i).split("_")[0]
    return y[:3] + "0s" if y.isdigit() and len(y) == 4 else "?"

style_cache = OUT / "style.json"
if style_cache.exists():
    STYLE = read_json(style_cache)
    print("loaded cached style table")
else:
    STYLE = {}
    for name, docs in SPLITS.items():
        seen, rows = set(), []
        for d in tqdm(docs, desc=f"style {name}", leave=False):
            h = text_key(d["text"])
            rows.append({"style": style(d["text"]), "decade": decade(d["id"]), "label": d["label"],
                         "newlines": "\n" in d["text"], "hyphen_wraps": d["text"].count("-\n"), "dup": h in seen})
            seen.add(h)
        STYLE[name] = rows
    style_cache.write_text(json.dumps(STYLE))

for name, rows in STYLE.items():
    df = pd.DataFrame(rows)
    u = df[~df["dup"]]
    print(f"\n===== {name}: {len(df):,} records -> {len(u):,} unique texts ({len(df) - len(u):,} exact duplicates)"
          f" | unique label-1 rate {u['label'].mean():.3f}")
    print(f"  style: { {k: int(v) for k, v in u["style"].value_counts().items()} } | with line breaks: {u['newlines'].mean():.1%}"
          f" | '-\\n' word wraps per case: {u['hyphen_wraps'].mean():.1f}")
    display(pd.crosstab(u["decade"], u["style"], values=u["label"], aggfunc=["count", "mean"]).round(3)
            .rename_axis(f"{name}: decade (count / rate of 1)"))


===== train: 37,387 records -> 31,652 unique texts (5,735 exact duplicates) | unique label-1 rate 0.414
  style: {'cased': 31641, 'lower': 11} | with line breaks: 0.0% | '-\n' word wraps per case: 0.0


count         mean      
style                               cased lower  cased lower
train: decade (count / rate of 1)                           
1940s                                99.0   NaN  0.455   NaN
1950s                               841.0   1.0  0.367  1.00
1960s                              2589.0   NaN  0.350   NaN
1970s                              3112.0   NaN  0.366   NaN
1980s                              2701.0   1.0  0.420  0.00
1990s                              7017.0   4.0  0.422  0.75
2000s                              9146.0   3.0  0.437  0.00
2010s                              5928.0   2.0  0.428  0.00
2020s                               208.0   NaN  0.413   NaN


===== dev: 1,988 records -> 994 unique texts (994 exact duplicates) | unique label-1 rate 0.500
  style: {'lower': 994} | with line breaks: 100.0% | '-\n' word wraps per case: 8.9


,count,mean
style,lower,lower
dev: decade (count / rate of 1),,
1950s,105,0.381
1960s,304,0.451
1970s,240,0.508
1980s,245,0.563
1990s,97,0.588
2000s,3,1.000



===== test: 1,517 records -> 1,517 unique texts (0 exact duplicates) | unique label-1 rate 0.502
  style: {'lower': 1500, 'cased': 17} | with line breaks: 98.9% | '-\n' word wraps per case: 9.0


count          mean       
style                            cased  lower  cased  lower
test: decade (count / rate of 1)                           
1950s                              NaN  134.0    NaN  0.463
1960s                              NaN  479.0    NaN  0.415
1970s                              NaN  381.0    NaN  0.486
1980s                              NaN  363.0    NaN  0.601
1990s                              4.0  134.0  1.000  0.590
2000s                              4.0    9.0  0.250  0.778
2010s                              9.0    NaN  0.778    NaN

## 12. Save summary


In [13]:
summary = {
    "format": FORMAT,
    "checks": checks.reset_index().to_dict(orient="records"),
    "label_counts": {n: dict(collections.Counter(d["label"] for d in SPLITS[n])) for n in LAB},
    "duplicate_groups": len(dup_groups), "duplicate_split_pairs": {"&".join(k): v for k, v in pair_counts.items()},
    "conflicting_duplicates": len(conflicting),
    "token_stats": rows, "epoch_estimates": est.to_dict(orient="index"),
    "length_baseline_dev": LENGTH_BASELINE,
    "naive_bayes_dev": NB["results"],
    "style": {n: {"records": len(r), "unique": sum(not x["dup"] for x in r),
                  "lower_unique": sum(x["style"] == "lower" and not x["dup"] for x in r)} for n, r in STYLE.items()},
    "tokenizer": str(TOK_PATH.relative_to(ROOT)),
}
(OUT / "data_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print("saved", (OUT / "data_summary.json").relative_to(ROOT))

saved experiments\phase4\p4_01_data\data_summary.json
